# Como foi feito o projeto de FP&A, passo a passo

**A pergunta de negócio.** A Atlântica Components (fictícia) está acima do orçamento no EBITDA de janeiro–setembro de 2026.
O CFO quer saber *porquê*, e se é uma boa notícia: melhor desempenho comercial, ou sorte nos preços, no calendário
e no câmbio que não se vai repetir.

**Como está organizado este notebook.** Segue o projeto pela ordem em que foi construído:

| Passo | O quê | Ferramenta |
|---|---|---|
| 1 | Gerar um razão geral em partidas dobradas, orçamentos e uma previsão revista, com histórias plantadas | Python (NumPy) |
| 2 | Carregar o razão e correr as verificações contabilísticas | pandas |
| 3 | Modelar a demonstração de resultados em dbt: convenção de sinais, conversão cambial, orçamento vs real | dbt sobre DuckDB |
| 4 | Responder às perguntas: orçamento vs real, ponte do EBITDA, câmbio, exceções no razão | pandas + Altair |
| 5 | Conferir o pandas com os marts do dbt | pandas + DuckDB |
| 6 | Publicar: painel web, Power BI e Qlik | Exportação JSON, PBIP, script Qlik |

**Os gráficos são interativos**: clique numa empresa para comandar os gráficos ligados e passe o cursor para ver os valores. São feitos com
[Altair](https://altair-viz.github.io/); abra a [versão interativa](https://guilhermerigonato.com/finance-fpa-analytics/notebook/pt/)
para os usar (o GitHub mostra os notebooks sem JavaScript). O código e os nomes das colunas ficam em inglês, como no repositório.

**Para correr:** `pip install pandas numpy altair duckdb dbt-duckdb`; depois, na raiz do repositório,
`python generator/generate.py` e `dbt build --profiles-dir .`, e execute todas as células (menos de um minuto).

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import altair as alt

RAW, SEEDS = Path("../data/raw"), Path("../seeds")
FY, LAST_CLOSED = 2026, pd.Timestamp("2026-09-01")      # variáveis do dbt current_fy e last_closed_month
pd.set_option("display.float_format", "{:,.0f}".format)

BLUE, ORANGE, GREEN, RED, GREY, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#d1453b", "#c9c8c3", "#52514e", "#e4e3df"

@alt.theme.register("portfolio", enable=True)
def portfolio_theme():
    axis = {"gridColor": GRID, "domainColor": MUTED, "tickColor": MUTED, "labelColor": MUTED,
            "titleColor": MUTED, "labelFontSize": 11, "titleFontSize": 11, "titleFontWeight": "normal"}
    return {"config": {"view": {"stroke": None}, "axis": axis, "legend": {"labelColor": MUTED, "titleColor": MUTED},
                       "title": {"anchor": "start", "fontSize": 13, "color": "#0b0b0b"},
                       "font": "system-ui, -apple-system, Segoe UI, sans-serif"}}

## Passo 1. Gerar os dados

O razão de um grupo real não pode ser publicado, por isso o [`generator/generate.py`](../generator/generate.py) escreve um: quatro
empresas (Portugal, Espanha e França em euros; o Reino Unido em libras), de janeiro de 2024 a setembro de 2026 fechado, um
razão geral em partidas dobradas, um auxiliar de vendas com unidades, taxas de câmbio mensais, os orçamentos de 2025 e 2026 e a
previsão revista 9+3 de outubro. Há oito histórias plantadas de propósito:

| # | História | Onde deve aparecer |
|---|---|---|
| 1 | A libra desvaloriza ao longo de 2026 (câmbio do orçamento 1,19 €/£) | Conversão cambial no Reino Unido |
| 2 | Nova transportadora de expedição em Portugal a partir de março | Transporte, picos no razão |
| 3 | As contratações comerciais em Espanha chegam meses atrasadas | Pessoal (favorável), volume (desfavorável) |
| 4 | Portugal sobe os preços de tabela 4% em janeiro | Preço de venda |
| 5 | As ferramentas, a linha de menor margem, são as que mais crescem | Mix de vendas |
| 6 | Uma fatura da CloudSuite lançada duas vezes no Reino Unido em junho | Verificação de faturas duplicadas |
| 7 | A França regulariza o bónus anual em dezembro de 2025 | Verificação de picos |
| 8 | Uma sobretaxa de 6% no aço dos rolamentos a partir de maio | Custo unitário |

A semente é fixa (`20261025`), por isso todas as execuções dão o mesmo razão.

In [2]:
gl = pd.read_csv(RAW / "gl_journal_lines.csv", dtype={"account_id": str}, parse_dates=["posting_date"])
plan = pd.read_csv(RAW / "plan_lines.csv", dtype={"account_id": str}, parse_dates=["month_start"])
vol = pd.read_csv(RAW / "sales_volume.csv", parse_dates=["month_start"])
fx = pd.read_csv(RAW / "fx_rates.csv", parse_dates=["month_start"])
accounts = pd.read_csv(SEEDS / "accounts.csv", dtype={"account_id": str})
entities = pd.read_csv(SEEDS / "entities.csv")
print(f"{len(gl):,} linhas de lançamento em {gl['journal_id'].nunique():,} lançamentos · {len(plan):,} linhas de plano · "
      f"{len(vol):,} linhas de volume de vendas · {len(fx)} taxas de câmbio")
gl.head(4)

81 072 linhas de lançamento em 40 536 lançamentos · 4536 linhas de plano · 1200 linhas de volume de vendas · 72 taxas de câmbio


,journal_id,line_number,entity_id,posting_date,account_id,cost_centre_id,product_line_id,amount_lc,source,vendor,document_ref
0,J0000001,1,PT01,2024-01-01,4000,SAL,BRG,"-27,390",AR,NaN,INV-PT01-240101-BRG
1,J0000001,2,PT01,2024-01-01,1100,NaN,NaN,"27,390",AR,NaN,INV-PT01-240101-BRG
2,J0000002,1,PT01,2024-01-01,5000,OPS,BRG,"16,966",INV,NaN,NaN
3,J0000002,2,PT01,2024-01-01,1300,NaN,NaN,"-16,966",INV,NaN,NaN


## Passo 2. Carregar o razão e correr as verificações contabilísticas

Um razão só serve se cumprir as partidas dobradas. Antes de qualquer análise: cada lançamento soma zero, cada conta existe
no plano de contas, cada empresa-mês tem taxa de câmbio e não há nada lançado depois do último mês fechado.

In [3]:
gl["month_start"] = gl["posting_date"].dt.to_period("M").dt.to_timestamp()
gl = gl.merge(entities[["entity_id", "currency"]], on="entity_id")
journal_totals = gl.groupby("journal_id")["amount_lc"].sum().round(2)
checks = {
    "every journal balances (debits = credits)": (journal_totals == 0).all(),
    "every account is in the chart": gl["account_id"].isin(accounts["account_id"]).all(),
    "every entity-month has an FX rate": gl[["currency", "month_start"]].drop_duplicates()
        .merge(fx, on=["currency", "month_start"], how="left")["avg_rate_eur"].notna().all(),
    "nothing posted after the last closed month": (gl["month_start"] <= LAST_CLOSED).all(),
    "the whole ledger sums to zero": round(gl["amount_lc"].sum(), 2) == 0,
}
assert all(checks.values()), checks
pd.Series(checks, name="passes").to_frame()

,passes
every journal balances (debits = credits),True
every account is in the chart,True
every entity-month has an FX rate,True
nothing posted after the last closed month,True
the whole ledger sums to zero,True


## Passo 3. Modelar a demonstração de resultados em dbt

O SQL está em [`models/`](../models). Três decisões moldam todos os números a seguir:

* **Convenção de sinais.** O razão guarda os débitos a positivo e os créditos a negativo. Os factos da demonstração de resultados invertem o
  sinal, por isso os proveitos são positivos, os custos negativos e *qualquer subtotal é uma soma simples*.
* **Duas conversões.** Os valores reais são convertidos à taxa média do mês (`amount_eur`) *e* à taxa do orçamento
  (`amount_eur_bud_rate`). A diferença entre as duas é o efeito cambial; tudo o resto é desempenho comercial.
* **Um único mart de resultados.** O `mart_pnl_monthly` põe lado a lado o real, o orçamento, a previsão 9+3 e o ano anterior por
  empresa, conta e mês, por isso todas as vistas (desvio, fecho previsto, ponte) leem uma só tabela.

Os marts da ponte (`mart_margin_pvm`, `mart_ebitda_bridge`) e as verificações do controller (`mart_journal_checks`) assentam
por cima. O `dbt build` corre 67 nós (modelos, seeds e testes: chaves, relações, valores aceites e um teste de que os
passos da ponte somam do EBITDA orçamentado ao real) sem falhas.

Em pandas, os mesmos factos:

In [4]:
pl_accounts = accounts[accounts["account_type"] == "P&L"].set_index("account_id")
rates = fx.set_index(["currency", "month_start"])

def translate(frame, budget=False):
    r = rates.loc[list(zip(frame["currency"], frame["month_start"]))]
    frame = frame.copy()
    frame["amount_eur"] = frame["amount"] * (r["budget_rate_eur"].values if budget else r["avg_rate_eur"].values)
    frame["amount_eur_bud_rate"] = frame["amount"] * r["budget_rate_eur"].values
    return frame

act = gl[gl["account_id"].isin(pl_accounts.index)].copy()
act["amount"] = -act["amount_lc"]                                   # proveitos +, custos -
act = translate(act.groupby(["entity_id", "currency", "account_id", "cost_centre_id", "product_line_id", "month_start"],
                            dropna=False)["amount"].sum().reset_index())

bud = plan[plan["scenario"] == f"BUD{FY}"].merge(entities[["entity_id", "currency"]], on="entity_id")
bud["amount"] = -bud["amount_lc"]
bud = translate(bud, budget=True)

ytd = lambda f: f[(f["month_start"].dt.year == FY) & (f["month_start"] <= LAST_CLOSED)]
ebitda_acc = pl_accounts.index[pl_accounts["pnl_subtotal"].isin(["Gross margin", "EBITDA"])]
e = lambda f: ytd(f[f["account_id"].isin(ebitda_acc)])
print(f"EBITDA do grupo jan–set {FY}: €{e(act)['amount_eur'].sum() / 1e6:,.2f}m real vs €{e(bud)['amount_eur'].sum() / 1e6:,.2f}m orçamento")

EBITDA do grupo jan–set 2026: 4,41 M€ real vs 4,17 M€ orçamento


## Passo 4. Responder às perguntas

### Que empresas estão à frente, e quando é que isso aconteceu?

Portugal está bem à frente, Espanha e França atrás, o Reino Unido ligeiramente atrás. A vista interativa disto, ligada à
ponte, aparece depois de a ponte ser calculada mais abaixo.

In [5]:
names = entities.set_index("entity_id")["entity_name"]
by_ent = pd.DataFrame({"actual": e(act).groupby("entity_id")["amount_eur"].sum(),
                       "budget": e(bud).groupby("entity_id")["amount_eur"].sum()})
by_ent["variance"] = by_ent["actual"] - by_ent["budget"]
by_ent["variance_pct"] = by_ent["variance"] / by_ent["budget"]
by_ent = by_ent.assign(entity=names).reset_index()

monthly = pd.concat([
    e(act).groupby(["entity_id", "month_start"])["amount_eur"].sum().rename("value").reset_index().assign(series="Actual"),
    e(bud).groupby(["entity_id", "month_start"])["amount_eur"].sum().rename("value").reset_index().assign(series="Budget"),
])
monthly = pd.concat([monthly, monthly.groupby(["month_start", "series"])["value"].sum().reset_index().assign(entity_id="GROUP")])

ent = alt.selection_point(name="ent", fields=["entity_id"], on="click", clear="dblclick")
chosen = "(ent.entity_id && length(ent.entity_id)) ? ent.entity_id[0] : 'GROUP'"

bars = alt.Chart(by_ent).mark_bar(cornerRadius=3).encode(
    y=alt.Y("entity:N", title=None, sort="x"),
    x=alt.X("variance_pct:Q", title="EBITDA vs orçamento, jan–set 2026", axis=alt.Axis(format="+%")),
    color=alt.condition("datum.variance < 0", alt.value(ORANGE), alt.value(BLUE)),
    opacity=alt.condition(ent, alt.value(1), alt.value(0.35)),
    tooltip=["entity", alt.Tooltip("actual:Q", format=",.0f"), alt.Tooltip("budget:Q", format=",.0f"),
             alt.Tooltip("variance:Q", format="+,.0f"), alt.Tooltip("variance_pct:Q", format="+.1%")],
).add_params(ent).properties(width=250, height=170, title="Clique numa empresa")

lines = alt.Chart(monthly).transform_filter(f"datum.entity_id == ({chosen})").mark_line(point=True).encode(
    x=alt.X("month_start:T", title=None, axis=alt.Axis(format="%b")),
    y=alt.Y("value:Q", title="EBITDA (€)", axis=alt.Axis(format="~s")),
    color=alt.Color("series:N", scale=alt.Scale(domain=["Actual", "Budget"], range=[BLUE, GREY]), title=None),
    strokeDash=alt.StrokeDash("series:N", scale=alt.Scale(domain=["Actual", "Budget"], range=[[1, 0], [5, 4]]), legend=None),
    tooltip=[alt.Tooltip("month_start:T", format="%b %Y"), "series", alt.Tooltip("value:Q", format=",.0f")],
).properties(width=420, height=170, title=alt.Title(alt.expr(f"'EBITDA mensal · ' + ({chosen})")))
by_ent[["entity", "actual", "budget", "variance", "variance_pct"]].style.format(
    {"actual": "{:,.0f}", "budget": "{:,.0f}", "variance": "{:+,.0f}", "variance_pct": "{:+.1%}"})

,entity,actual,budget,variance,variance_pct
0,Atlántica España,"1,194,708","1,366,294","-171,586",-12.6%
1,Atlantique France,"297,814","344,171","-46,357",-13.5%
2,Atlântica Portugal,"2,266,729","1,762,878","+503,852",+28.6%
3,Atlantica UK Ltd,"649,301","698,240","-48,939",-7.0%


### Porquê: a ponte do EBITDA

Uma ponte vai do EBITDA orçamentado ao real em passos sobre os quais um CFO pode agir. A margem de produto divide-se em **volume, mix,
preço e custo unitário**; os outros passos são serviços e descontos, transporte, pessoal, outros gastos operacionais e **conversão
cambial**, medida à parte para que a moeda não se esconda nas linhas comerciais. As linhas de produto vendem-se em unidades muito
diferentes (um parafuso não é uma bomba), por isso o volume mede-se como unidades reais a preço do orçamento, o que dá a todas
as linhas uma unidade comum:

* volume = variação das vendas a preços do orçamento × taxa de margem orçamentada da empresa
* mix = a mesma variação à taxa de margem orçamentada de cada linha, menos o efeito volume
* preço = unidades reais × (preço real − preço do orçamento), à taxa do orçamento
* custo unitário = unidades reais × (custo unitário do orçamento − custo unitário real), à taxa do orçamento

In [6]:
# Auxiliar de vendas: unidades e vendas por cenário, convertidas
vs = vol.merge(entities[["entity_id", "currency"]], on="entity_id")
vs = vs[(vs["month_start"].dt.year == FY) & (vs["month_start"] <= LAST_CLOSED) & vs["scenario"].isin(["ACT", f"BUD{FY}"])]
r = rates.loc[list(zip(vs["currency"], vs["month_start"]))]
vs["rev_eur"] = vs["revenue_lc"] * np.where(vs["scenario"] == "ACT", r["avg_rate_eur"].values, r["budget_rate_eur"].values)
vs["rev_bud_rate"] = vs["revenue_lc"] * r["budget_rate_eur"].values
s = vs.groupby(["scenario", "entity_id", "product_line_id"])[["units", "rev_eur", "rev_bud_rate"]].sum()
a_, b_ = s.loc["ACT"], s.loc[f"BUD{FY}"]
cost = lambda f, col: -ytd(f[f["account_id"] == "5000"]).groupby(["entity_id", "product_line_id"])[col].sum()

l = pd.DataFrame({"act_units": a_["units"], "act_rev": a_["rev_eur"], "act_rev_bud_rate": a_["rev_bud_rate"],
                  "bud_units": b_["units"], "bud_rev": b_["rev_eur"],
                  "act_cost": cost(act, "amount_eur"), "act_cost_bud_rate": cost(act, "amount_eur_bud_rate"),
                  "bud_cost": cost(bud, "amount_eur")})
l["bud_price"] = l["bud_rev"] / l["bud_units"]
l["bud_unit_cost"] = l["bud_cost"] / l["bud_units"]
l["bud_margin_rate"] = (l["bud_rev"] - l["bud_cost"]) / l["bud_rev"]
l["act_at_bud_price"] = l["act_units"] * l["bud_price"]
g = l.groupby(level="entity_id")
l["ent_margin_rate"] = (g["bud_rev"].transform("sum") - g["bud_cost"].transform("sum")) / g["bud_rev"].transform("sum")
l["volume"] = (l["act_at_bud_price"] - l["bud_rev"]) * l["ent_margin_rate"]
l["mix"] = (l["act_at_bud_price"] - l["bud_rev"]) * (l["bud_margin_rate"] - l["ent_margin_rate"])
l["price"] = l["act_rev_bud_rate"] - l["act_at_bud_price"]
l["unit_cost"] = l["act_units"] * l["bud_unit_cost"] - l["act_cost_bud_rate"]
pvm = l.groupby(level="entity_id")[["volume", "mix", "price", "unit_cost"]].sum()

# O resto da ponte sai da demonstração de resultados, à taxa do orçamento
grp = pl_accounts["pnl_group"]
def var(accts):
    a = e(act)[e(act)["account_id"].isin(accts)].groupby("entity_id")["amount_eur_bud_rate"].sum()
    b = e(bud)[e(bud)["account_id"].isin(accts)].groupby("entity_id")["amount_eur"].sum()
    return a.sub(b, fill_value=0)

steps = pd.DataFrame({
    "Budget EBITDA": e(bud).groupby("entity_id")["amount_eur"].sum(),
    "Sales volume": pvm["volume"], "Sales mix": pvm["mix"], "Sales price": pvm["price"],
    "Services and discounts": var(["4100", "4900"]),
    "Unit cost": pvm["unit_cost"],
    "Freight": var(["5100", "5200"]),
    "People": var(grp.index[grp == "People"]),
    "Other opex": var(grp.index[grp.isin(["Facilities", "IT", "Commercial", "General and admin"])]),
    "FX translation": e(act).groupby("entity_id")["amount_eur"].sum() - e(act).groupby("entity_id")["amount_eur_bud_rate"].sum(),
    "Actual EBITDA": e(act).groupby("entity_id")["amount_eur"].sum(),
}).fillna(0)
steps.loc["GROUP"] = steps.sum()
inner = steps.columns[1:-1]
assert np.allclose(steps["Budget EBITDA"] + steps[inner].sum(axis=1), steps["Actual EBITDA"]), "the bridge must close"
(steps.T / 1e3).round(0).rename_axis("€ thousand")

entity_id,ES01,FR01,PT01,UK01,GRUPO
milhares de €,,,,,
EBITDA orçamentado,"1,366",344,"1,763",698,"4,172"
Volume de vendas,-72,51,25,47,51
Mix de vendas,-41,-36,-71,-25,-173
Preço de venda,-10,0,862,15,868
Serviços e descontos,-109,14,-16,-12,-124
Custo unitário,-86,-69,-115,-53,-324
Transporte,8,-6,-169,-4,-171
Pessoal,143,-3,-2,3,141
Outros gastos,-4,2,-10,4,-8


In [7]:
# Cascata: uma linha por empresa e passo, com início e fim.
# As barras das empresas, as linhas mensais e a cascata partilham uma seleção, por isso são desenhadas como um só gráfico:
# clique numa empresa em cima à esquerda e as linhas mensais e a ponte acompanham; duplo clique para o grupo.
rows = []
for ent_id, row in steps.iterrows():
    run = 0.0
    for i, (step, v) in enumerate(row.items()):
        total = step in ("Budget EBITDA", "Actual EBITDA")
        start, end = (0, v) if total else (run, run + v)
        run = v if total else run + v
        rows.append(dict(entity_id=ent_id, order=i, step=step, start=start, end=end, amount=v,
                         kind="Total" if total else ("Up" if v >= 0 else "Down")))
wf = pd.DataFrame(rows)

waterfall = alt.Chart(wf).transform_filter(f"datum.entity_id == ({chosen})").mark_bar(size=34).encode(
    x=alt.X("step:N", sort=list(steps.columns), title=None, axis=alt.Axis(labelAngle=-30, labelLimit=140)),
    y=alt.Y("start:Q", title="€", axis=alt.Axis(format="~s")), y2="end:Q",
    color=alt.Color("kind:N", scale=alt.Scale(domain=["Total", "Up", "Down"], range=[GREY, GREEN, RED]), legend=None),
    tooltip=["step", alt.Tooltip("amount:Q", format="+,.0f")],
).properties(width=700, height=280, title=alt.Title(alt.expr(f"'Ponte do EBITDA, do orçamento ao real · ' + ({chosen})")))
alt.vconcat(alt.hconcat(bars, lines), waterfall).resolve_scale(color="independent")

alt.VConcatChart(...)

**Como ler a ponte.** No grupo, o *preço de venda* é o maior passo positivo, e quase todo vem da subida de 4% nos preços de tabela
em Portugal (história 4). O custo unitário (a sobretaxa do aço, história 8), o transporte (a transportadora de Portugal, história 2) e o mix
(as ferramentas a crescer, história 5) puxam no sentido contrário. Sem a subida de preços em Portugal, o grupo estaria abaixo do orçamento.
O passo *Pessoal* de Espanha só é favorável porque as contratações chegaram tarde (história 3), enquanto o volume e os serviços ficam aquém.

### Quanto do desvio do Reino Unido é câmbio?

O orçamento do Reino Unido foi feito a 1,19 € por £. Converter os valores reais do Reino Unido à taxa do orçamento, em vez da média do mês,
separa o desempenho comercial do câmbio. Passe o cursor pelas linhas para ver as taxas mensais.

In [8]:
gbp = fx[(fx["currency"] == "GBP") & (fx["month_start"].dt.year >= 2025)].melt(
    id_vars="month_start", value_vars=["avg_rate_eur", "budget_rate_eur"], var_name="rate", value_name="eur_per_gbp")
gbp["rate"] = gbp["rate"].map({"avg_rate_eur": "Monthly average", "budget_rate_eur": "Budget rate"})
uk = ytd(act[(act["entity_id"] == "UK01") & (act["account_id"] == "4000")])
uk_b = ytd(bud[(bud["entity_id"] == "UK01") & (bud["account_id"] == "4000")])
print(f"Vendas de produto do Reino Unido vs orçamento: {uk['amount_eur_bud_rate'].sum() / uk_b['amount_eur'].sum() - 1:+.1%} à taxa do orçamento, "
      f"{uk['amount_eur'].sum() / uk_b['amount_eur'].sum() - 1:+.1%} em euros às taxas reais; "
      f"o câmbio custou €{(uk['amount_eur_bud_rate'].sum() - uk['amount_eur'].sum()) / 1e3:,.0f}k de vendas")
alt.Chart(gbp).mark_line(point=True).encode(
    x=alt.X("month_start:T", title=None), y=alt.Y("eur_per_gbp:Q", title="€ por £", scale=alt.Scale(zero=False)),
    color=alt.Color("rate:N", scale=alt.Scale(range=[BLUE, GREY]), title=None),
    tooltip=[alt.Tooltip("month_start:T", format="%b %Y"), "rate", alt.Tooltip("eur_per_gbp:Q", format=".3f")],
).properties(width=560, height=200, title="Libra face à taxa do orçamento")

Vendas de produto do Reino Unido vs orçamento: +1,6% à taxa do orçamento, −2,0% em euros às taxas reais; o câmbio custou 362 k€ de vendas


alt.Chart(...)

### O que deve o controller resolver antes do fecho?

Duas verificações que um controller corre sobre o próprio razão:

* **Faturas duplicadas:** a mesma empresa, fornecedor, referência do documento e valor lançados mais de uma vez.
* **Picos:** uma conta de uma empresa num mês que variou mais de 25% face *tanto* aos três meses anteriores como ao mesmo
  mês do ano anterior (e em mais de 2000), o que ignora a sazonalidade normal.

Passe o cursor por um ponto para ver o detalhe.

In [9]:
first = gl[(gl["line_number"] == 1) & gl["document_ref"].notna() & (gl["amount_lc"] > 0)]
dups = (first.groupby(["entity_id", "account_id", "vendor", "document_ref", "amount_lc"])
             .agg(postings=("journal_id", "size"), month_start=("month_start", "min")).reset_index())
dups = dups[dups["postings"] > 1]

cost_lines = gl[gl["account_id"].isin(["5100", "5200"]) | gl["account_id"].str.startswith("6")]
m = cost_lines.groupby(["entity_id", "account_id", "month_start"])["amount_lc"].sum().mul(-1).rename("amount").reset_index()
m = m.sort_values(["entity_id", "account_id", "month_start"])
g = m.groupby(["entity_id", "account_id"])["amount"]
m["avg_3m"] = g.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
m["same_month_ly"] = g.shift(12)
spikes = m[m["same_month_ly"].notna() & ((m["amount"] / m["avg_3m"] - 1).abs() > 0.25)
           & ((m["amount"] / m["same_month_ly"] - 1).abs() > 0.25) & ((m["amount"] - m["same_month_ly"]).abs() > 2000)].copy()
spikes["change_vs_ly"] = spikes["amount"] / spikes["same_month_ly"] - 1
spikes["account"] = spikes["account_id"].map(pl_accounts["account_name"])
print(f"{len(dups)} fatura(s) duplicada(s): " + "; ".join(f"{r.entity_id} {r.vendor} {r.document_ref} ({r.postings} lançamentos)" for r in dups.itertuples()))

alt.Chart(spikes).mark_circle(size=140).encode(
    x=alt.X("month_start:T", title=None),
    y=alt.Y("change_vs_ly:Q", title="Variação face ao mesmo mês do ano anterior", axis=alt.Axis(format="+%", values=[-0.5, 0, 0.5, 1, 2, 4, 6]), scale=alt.Scale(type="symlog")),
    color=alt.Color("entity_id:N", title="Empresa", scale=alt.Scale(range=[BLUE, ORANGE, GREEN, "#9a6fd0"])),
    tooltip=["entity_id", "account", alt.Tooltip("month_start:T", format="%b %Y"),
             alt.Tooltip("change_vs_ly:Q", format="+.0%"), alt.Tooltip("amount:Q", title="Custo (moeda local)", format=",.0f")],
).properties(width=560, height=220, title="Picos de custo assinalados para revisão")

1 fatura(s) duplicada(s): UK01 CloudSuite Ltd CLO-UK01-2606-0 (2 lançamentos)


alt.Chart(...)

As verificações encontram os casos plantados e mais nada: a fatura da CloudSuite lançada duas vezes no Reino Unido em junho (e o seu
estorno em agosto aparece como pico), o transporte de expedição de Portugal a partir de março e a regularização do bónus de dezembro de 2025
em França.

## Passo 5. Conferir o pandas com os marts do dbt

A ponte, a decomposição da margem e as verificações foram construídas duas vezes, em SQL e em pandas. Têm de coincidir ao cêntimo.

In [10]:
db = duckdb.connect("../data/fpa.duckdb", read_only=True)
m_bridge = db.sql("select entity_id, step, amount_eur from mart_ebitda_bridge").df().pivot(index="entity_id", columns="step", values="amount_eur")
m_checks = db.sql("select check_type, count(*) n from mart_journal_checks group by all").df().set_index("check_type")["n"]
db.close()

ents = ["PT01", "ES01", "FR01", "UK01"]
diff = (steps.loc[ents, m_bridge.columns] - m_bridge.loc[ents]).abs().max().max()
recon = pd.DataFrame([
    ["Ponte: maior diferença em qualquer empresa e passo (€)", diff, 0],
    ["EBITDA real do grupo (€)", steps.loc["GROUP", "Actual EBITDA"], m_bridge["Actual EBITDA"].sum()],
    ["Efeito preço de venda do grupo (€)", steps.loc["GROUP", "Sales price"], m_bridge["Sales price"].sum()],
    ["Conversão cambial do Reino Unido (€)", steps.loc["UK01", "FX translation"], m_bridge.loc["UK01", "FX translation"]],
    ["Faturas duplicadas", len(dups), m_checks.get("duplicate_invoice", 0)],
    ["Picos", len(spikes), m_checks.get("spike", 0)],
], columns=["measure", "pandas", "dbt"])
recon["match"] = np.isclose(recon["pandas"].astype(float), recon["dbt"].astype(float), atol=0.01)
assert recon["match"].all(), recon
recon.style.format({"pandas": "{:,.2f}", "dbt": "{:,.2f}"})

,measure,pandas,dbt,match
0,Ponte: maior diferença em qualquer empresa e passo (€),0.00,0.00,True
1,EBITDA real do grupo (€),"4,408,553.06","4,408,553.06",True
2,Efeito preço de venda do grupo (€),"868,077.19","868,077.19",True
3,Conversão cambial do Reino Unido (€),"-23,901.88","-23,901.88",True
4,Faturas duplicadas,1.00,1.00,True
5,Picos,6.00,6.00,True


Cada passo da ponte, em cada empresa, coincide com o mart do dbt ao cêntimo, e as duas verificações encontram os mesmos casos.

## Passo 6. Publicar

* **Painel web** ([`web/`](../web), online em [guilhermerigonato.com/finance-fpa-analytics](https://guilhermerigonato.com/finance-fpa-analytics/?lang=pt)):
  [`scripts/export.py`](../scripts/export.py) escreve JSON a partir dos marts; clicar numa empresa, num mês, num centro de custo
  ou numa linha do razão filtra os outros painéis.
* **Power BI** ([`powerbi/`](../powerbi)): PBIP com modelo semântico em TMDL, medidas DAX para o desvio e a
  ponte, segurança ao nível da linha por empresa, verificado com as regras do Best Practice Analyzer em [`bpa-rules.json`](../bpa-rules.json).
* **Qlik Sense** ([`qlik/`](../qlik)): script de carga, set analysis para orçamento vs real, section access por empresa.

## Que biblioteca usar para gráficos interativos

| Biblioteca | Gráficos ligados | Resultado | Melhor para |
|---|---|---|---|
| **Altair** (Vega-Lite), usada aqui | Nativos: seleções partilhadas entre gráficos | HTML estático, sem servidor | Relatórios que têm de abrir em qualquer lado |
| **Plotly** | Por gráfico; ligar gráficos exige Dash | HTML; uma app Dash para ligar | Gráficos individuais ricos, mapas |
| **Bokeh** | Seleção ligada através de uma fonte de dados partilhada | HTML ou um servidor Bokeh | Dispersões grandes, ferramentas à medida |
| **HoloViews + Panel** | `link_selections` numa só chamada | Notebook ou app Panel | Transformar um notebook numa pequena app |
| **Dash / Streamlit** | Qualquer, com callbacks em Python | Precisa de servidor | Ferramentas internas com dados ao vivo |